# Step 05 — Prompt Template

🇬🇧 **English** (this notebook) · 🇩🇪 Deutsch — not yet available.

Same question as Step 03, but the prompt is split into two roles — `system` (who the model is, how it should behave) and `user` (the actual question). Try removing individual components to see what each one actually does.

## Learning objective

By the end of this notebook, you will:

- Understand the difference between the `system` and `user` roles in a chat-model API call
- Have built a reusable system-message template out of named components (persona, instruction, context, format, audience, tone)
- Be able to identify which single component drives most of the change in a model's output

## Prerequisites

- [Step 03 — Zero-Shot Prompting](step_03_zero_shot_prompting.ipynb) completed, with your team's topic already chosen
- The same `.env` setup as the previous steps

## Background

Every message you send a chat model has a **role**. `user` is the question; `system` is background instructions the end user never sees, but which shape every response — who the model is, what it knows, how it should behave. This `system`/`user` split is a first-class part of the Chat Completions API most providers implement, not just a formatting convention.

Splitting a prompt into named components (persona, instruction, context, format, audience, tone) is one specific way of writing a `system` message — turning an unstructured block of instructions into named, independently-adjustable parts:

> Liu, P., et al. (2023). *Pre-train, Prompt, and Predict: A Systematic Survey of Prompting Methods in Natural Language Processing*. ACM Computing Surveys, 55(9), 1–35. [arXiv:2107.13586](https://arxiv.org/abs/2107.13586)

## How this works

Run the Setup cell once, then both examples:

1. **Setup** loads your API keys and creates the `llm` object, same as before.
2. **Example 1 and Example 2** each build a `system_message` out of six named string variables — `persona`, `instruction`, `context`, `data_format`, `audience`, `tone` — concatenated together, then sent as the `system` role alongside a `user` message holding the real question.
3. Both examples ask the *exact same question* — only the six `system_message` values differ. Any difference in the output has to come from the system message alone.

In [1]:
import os

# CrewAI's telemetry tries to reach its backend over the network on import;
# on a restricted/firewalled connection this can hang for a long time with no
# error. Disable it before crewai is imported.
os.environ.setdefault('CREWAI_DISABLE_TELEMETRY', 'true')

from dotenv import load_dotenv
from crewai import LLM
from IPython.display import Markdown, display

load_dotenv()

llm = LLM(model=os.getenv("MODEL", "gemini/gemini-3.1-flash-lite"))

### Example 1 — Teacher persona

Explains the topic warmly and simply, for a young audience with no background.

In [2]:
# ── System message — who the model is and how it should behave ───────────────
# These components are sent as background instructions, not as a question.
persona       = "You are a helpful assistant.\n"
instruction   = "You are an expert in EU AI Act compliance.\n"
context       = "You are explaining this to someone with no legal or technical background.\n"
data_format   = "Answer in a single short paragraph.\n"
audience      = "The reader is a curious 10-year-old.\n"
tone          = "Be warm, simple, and encouraging.\n"

system_message = persona + instruction + context + data_format + audience + tone

# ── User message — the actual question ───────────────────────────────────────
user_message = "Explain the EU AI Act in simple terms in a short paragraph."

output = llm.call(
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]
)
display(Markdown(output))

Think of the EU AI Act like a safety manual or a set of "playground rules" for robots and computer brains. Just like we have rules to make sure toys are safe to play with, this law makes sure that companies building smart technology keep people safe, treat everyone fairly, and don't keep secrets about how their inventions work. It’s basically the EU’s way of making sure that as technology gets smarter, it stays helpful and kind to all of us!

### Example 2 — Engineer persona

Same topic, same question, same six component names — only the values change. Compare the tone, vocabulary, and depth against Example 1.

In [3]:
# ── System message — same six components, different values ───────────────────
persona       = "You are a senior machine learning engineer.\n"
instruction   = "You are reviewing the compliance implications of the EU AI Act for your product.\n"
context       = "You are explaining this to a fellow engineer who already works with AI systems.\n"
data_format   = "Answer in a single short paragraph.\n"
audience      = "The reader is assessing what needs to change in the product.\n"
tone          = "Be concise and matter-of-fact.\n"

system_message = persona + instruction + context + data_format + audience + tone

# ── User message — the same question as Example 1 ─────────────────────────────
user_message = "Explain the EU AI Act in simple terms in a short paragraph."

output = llm.call(
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]
)
display(Markdown(output))

The EU AI Act classifies systems by risk level, imposing strict obligations on "high-risk" applications—such as those used in critical infrastructure, recruitment, or biometric identification—which now require rigorous risk management, data governance, transparency, and human oversight. You must audit your product to determine its risk tier, implement mandatory technical documentation, ensure logging and traceability for all automated decisions, and establish clear quality management systems to maintain compliance before deployment. If your system interacts with users or processes sensitive data, you will also face new transparency requirements regarding model disclosures and the mitigation of inherent algorithmic biases.

## Your task

1. Run the cells in order (Setup, then Example 1, then Example 2).

2. Compare Example 1 and Example 2 — same question, same topic, only the system message differs. What changed: vocabulary? Structure? Depth? Which single component do you think drove that most?

3. Pick one component in either example (e.g. `tone`) and change just that one, leaving everything else the same. Re-run. Does changing one component shift the output as much as the full persona swap between Example 1 and 2 did?

4. Note what you observed — this is exactly the kind of reasoning `REPORT.md`'s Section 4.3 (Agent Identity) asks you to show for your own agents' `role`/`goal`/`backstory`.

## Shortcomings

A prompt template still produces one answer in one pass — if the task genuinely needs multiple distinct stages (e.g. gather facts, *then* explain them to a specific audience), cramming all of that into a single `system` message asks one model call to do too much at once, and the components start fighting each other rather than cooperating.

[Step 06](step_06_chain_prompting.ipynb) addresses exactly that: splitting a task into two sequential calls instead of one bigger prompt.

## Resources for further reading

- Liu, P., et al. (2023). *Pre-train, Prompt, and Predict: A Systematic Survey of Prompting Methods in Natural Language Processing*. ACM Computing Surveys, 55(9), 1–35. [arXiv:2107.13586](https://arxiv.org/abs/2107.13586)
- [OpenAI Chat Completions API guide](https://platform.openai.com/docs/guides/text-generation) — the `system`/`user`/`assistant` role structure most providers, including Gemini via `litellm`, follow.